In [ ]:
# Cell 1: Imports and Setup
import pandas as pd
import sqlite3
import os
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# --- Configuration ---
# ASSUMPTION: The Kaggle file 'database.sqlite' is in the current directory.
# Corrected DB_PATH to point to the downloaded file location
DB_PATH = '/kaggle/input/soccer/database.sqlite'
print("Setup complete. Ready to load database tables.")

Setup complete. Ready to load database tables.


### Imports and Setup

This cell imports the necessary libraries for data manipulation, database interaction, and machine learning. It also sets up the path to the SQLite database file.

### Kaggle Credentials Setup

This cell reads your Kaggle API credentials from the `kaggle.json` file you uploaded, and sets them as environment variables. This allows the notebook to interact with the Kaggle API to download datasets.

In [ ]:
import json
import os

# Path to the uploaded kaggle.json file
kaggle_json_path = 'kaggle.json'

# Read the credentials from the JSON file
with open(kaggle_json_path, 'r') as f:
    kaggle_creds = json.load(f)

# Set environment variables
os.environ['KAGGLE_USERNAME'] = kaggle_creds['username']
os.environ['KAGGLE_KEY'] = kaggle_creds['key']

print("Kaggle credentials loaded and environment variables set.")

Kaggle credentials loaded and environment variables set.


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hugomathien/soccer")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'soccer' dataset.
Path to dataset files: /kaggle/input/soccer


In [ ]:
# Cell 2: Load DataFrames from SQLite

def load_esd_tables(db_path):
    """Loads necessary tables from the European Soccer Database (SQLite)."""
    try:
        conn = sqlite3.connect(db_path)
    except sqlite3.OperationalError:
        print(f"ERROR: Could not find database file at '{db_path}'. Please check the DB_PATH variable.")
        return None, None, None, None

    # 1. Match Data: Fixtures, scores, and starting XI IDs (Ground Truth source)
    match_df = pd.read_sql_query("SELECT * FROM Match", conn)

    # 2. Player Attributes: FIFA-sourced player ratings and skills (Feature source)
    player_attributes_df = pd.read_sql_query("SELECT * FROM Player_Attributes", conn)

    # 3. Player Info: Names and general player IDs
    player_df = pd.read_sql_query("SELECT * FROM Player", conn)

    # 4. Team Info: Team names
    team_df = pd.read_sql_query("SELECT * FROM Team", conn)

    conn.close()

    print(f"Loaded {match_df.shape[0]:,} matches for Transfer Learning.")
    print(f"Loaded {player_attributes_df.shape[0]:,} player attribute records.")

    return match_df, player_attributes_df, player_df, team_df

# *** EXECUTION: Run the loading function ***
match_data_df, player_att_df, player_info_df, team_info_df = load_esd_tables(DB_PATH)

Loaded 25,979 matches for Transfer Learning.
Loaded 183,978 player attribute records.


In [ ]:
# Cell 3: Action 1.2 - Generate Ground Truth (Y) and Match Instances (FINAL MEMORY-SAFE VERSION)

def create_selection_ground_truth_optimized(match_df, team_df, db_path, negative_sample_ratio=5):
    """
    Generates positive and memory-safe negative samples by:
    1. Sampling a small subset of matches (e.g., a few top seasons).
    2. Using a small, safe negative sampling ratio (e.g., 5:1).
    """

    # --- STEP 0: EXTREME DATA FILTERING (To prevent RAM crash) ---
    # We will limit the matches to a few top leagues and a few recent seasons available in ESD (2012-2016)

    # 1. Get IDs for top leagues (England, Spain, Germany, Italy)
    top_league_names = ['England', 'Spain', 'Germany', 'Italy']
    # Use the db_path passed into the function for the connection
    conn = sqlite3.connect(db_path)
    country_df = pd.read_sql_query("SELECT * FROM Country", conn)
    league_df = pd.read_sql_query("SELECT * FROM League", conn)
    conn.close()

    target_country_ids = country_df[country_df['name'].isin(top_league_names)]['id'].tolist()

    # 2. Filter the Match Data Frame
    match_df_filtered = match_df[
        (match_df['country_id'].isin(target_country_ids)) &
        (match_df['season'].isin(['2012/2013', '2013/2014', '2014/2015', '2015/2016']))
    ].copy()

    print(f"Reduced match count to {match_df_filtered.shape[0]:,} for memory safety.")

    # Use the filtered DF for all subsequent steps
    match_lineups_df = match_df_filtered[['id', 'season', 'date', 'home_team_api_id', 'away_team_api_id'] +
                                       [col for col in match_df_filtered.columns if 'player_' in col and 'X' not in col and 'Y' not in col]].copy()

    positive_samples_list = []

    # --- 1. Generate POSITIVE Samples (Y=1) ---
    def collect_positive_samples(row, team_prefix, team_id_col, is_home):
        player_id_cols_subset = [col for col in match_df_filtered.columns if team_prefix in col and 'player_' in col and 'X' not in col and 'Y' not in col]
        for player_col in player_id_cols_subset:
            player_id = row[player_col]
            if pd.notna(player_id):
                positive_samples_list.append({
                    'match_api_id': row['id'],
                    'player_api_id': int(player_id),
                    'team_api_id': row[team_id_col],
                    'is_home': is_home,
                    'season': row['season'],
                    'date': row['date'],
                    'selected_in_xi': 1
                })

    match_lineups_df.apply(lambda row: collect_positive_samples(row, 'home_', 'home_team_api_id', 1), axis=1)
    match_lineups_df.apply(lambda row: collect_positive_samples(row, 'away_', 'away_team_api_id', 0), axis=1)

    positive_samples_df = pd.DataFrame(positive_samples_list).drop_duplicates(
        subset=['match_api_id', 'player_api_id', 'team_api_id'], keep='first'
    )

    # --- 2. Generate NEGATIVE Samples (Y=0) - The Non-Selected (Memory-Safe Sampling) ---

    all_unique_players = positive_samples_df['player_api_id'].unique()
    negative_samples_list = []

    # Use a dictionary for fast lookup of players who DID start in a given match/team combo
    positive_samples_df['key'] = positive_samples_df['match_api_id'].astype(str) + '_' + positive_samples_df['team_api_id'].astype(str)

    for index, row in match_df_filtered.iterrows(): # Use the filtered matches
        match_id = row['id']
        teams = [row['home_team_api_id'], row['away_team_api_id']]

        # Use a single list of all players selected in this match (across both teams) for quick filtering later
        all_starters_in_match = positive_samples_df[
             (positive_samples_df['match_api_id'] == match_id)
        ]['player_api_id'].unique()

        for team_id in teams:
            if pd.isna(team_id): continue

            # Find players NOT selected for this match
            non_selected_players = np.setdiff1d(all_unique_players, all_starters_in_match)

            # Sample a fixed number of negative examples (e.g., 5 times the starting XI)
            # 11 starters * 5 = 55 negative samples per team/match
            num_to_sample = int(11 * negative_sample_ratio)

            if len(non_selected_players) > num_to_sample:
                sampled_players = np.random.choice(non_selected_players, num_to_sample, replace=False)
            else:
                sampled_players = non_selected_players

            # Create the negative samples for this match/team
            for player_id in sampled_players:
                negative_samples_list.append({
                    'match_api_id': match_id,
                    'player_api_id': int(player_id),
                    'team_api_id': int(team_id),
                    'season': row['season'],
                    'date': row['date'],
                    'selected_in_xi': 0 # NEGATIVE CLASS
                })

    negative_samples_df = pd.DataFrame(negative_samples_list)
    positive_samples_df = positive_samples_df.drop(columns=['key']) # Clean up key


    # --- 3. Combine POSITIVE and sampled NEGATIVE samples ---
    player_match_instances = pd.concat([positive_samples_df, negative_samples_df], ignore_index=True)

    # Final Merge for Team Name
    player_match_instances = pd.merge(
        player_match_instances,
        team_df[['team_api_id', 'team_long_name']],
        on='team_api_id',
        how='left'
    )

    # Final sanity check on the target variable Y
    player_match_instances['selected_in_xi'] = player_match_instances['selected_in_xi'].astype(int)

    print(f"\nFinal Training Samples created: {player_match_instances.shape[0]:,}")
    print(f"Target Y distribution (1=Selected, 0=Not Selected):")
    print(player_match_instances['selected_in_xi'].value_counts())

    return player_match_instances

# *** EXECUTION ***
# The name has changed to match the function:
player_match_instances_df = create_selection_ground_truth_optimized(match_data_df, team_info_df, DB_PATH)

Reduced match count to 5,783 for memory safety.

Final Training Samples created: 763,161
Target Y distribution (1=Selected, 0=Not Selected):
selected_in_xi
0    636130
1    127031
Name: count, dtype: int64


In [ ]:
# Cell 4: Generate Synthetic Fitness Features (ACWR Proxy) - MERGE-BASED FIX

def generate_fitness_features(df):
    """
    Calculates Acute:Chronic Workload Ratio (ACWR) using Minutes Played as proxy.
    This method avoids complex groupby().apply() indexing issues by using
    a unique date index and merging the results back.
    """

    # 1. Prepare Base Data
    df['minutes_load'] = 90
    df['date'] = pd.to_datetime(df['date'])

    # Ensure a unique key for merging results later: player_match_key
    df['player_match_key'] = df['player_api_id'].astype(str) + '_' + df['match_api_id'].astype(str)

    # --- 2. Calculate Load for EACH Player using Rolling Window on a Temp DF ---

    # Get a list of all unique players
    unique_players = df['player_api_id'].unique()

    load_results = []

    for player_id in unique_players:
        # Filter all historical data for this single player
        player_df = df[df['player_api_id'] == player_id].sort_values('date')

        # Create a temporary DataFrame with 'date' as the index for time rolling
        temp_load_df = player_df[['date', 'minutes_load', 'player_match_key']].copy()
        temp_load_df = temp_load_df.set_index('date')

        # Calculate Acute Load (AL): Rolling sum over the last 7 days (Fatigue)
        # We use a shift(-1) to achieve 'closed=left' (data BEFORE the current match)
        temp_load_df['Acute_Load'] = temp_load_df['minutes_load'].rolling(window='7D').sum().shift(1).fillna(0)

        # Chronic Load (CL): Rolling mean over the last 28 days (Fitness)
        temp_load_df['Chronic_Load'] = temp_load_df['minutes_load'].rolling(window='28D').mean().shift(1).fillna(0)

        # Select the key results and append
        load_results.append(temp_load_df[['Acute_Load', 'Chronic_Load', 'player_match_key']].reset_index(drop=True))


    # 3. Combine all player results and merge back to the main DataFrame
    load_features = pd.concat(load_results, ignore_index=True)

    # Merge the calculated features (AL, CL) back onto the main DF using the unique key
    df = pd.merge(
        df,
        load_features,
        on='player_match_key',
        how='left'
    )

    # 4. Calculate ACWR
    epsilon = 0.001
    df['ACWR'] = df['Acute_Load'] / (df['Chronic_Load'] + epsilon)
    df['ACWR'] = df['ACWR'].clip(upper=3.0)

    # Final cleanup of temporary and original load columns
    df = df.drop(columns=['player_match_key', 'minutes_load'])

    print("Generated ACWR (Fitness Proxy) feature successfully using merge-based calculation.")
    return df

# *** EXECUTION ***
player_match_instances_with_fitness_df = generate_fitness_features(player_match_instances_df)

Generated ACWR (Fitness Proxy) feature successfully using merge-based calculation.


Explanation of Cell 10 (ACWR Fitness Feature)
This cell generates the Consistency/Fitness feature required by the abstract using the Acute:Chronic Workload Ratio (ACWR).

Workload Proxy: It assumes each match instance represents 90 minutes of load.

Acute/Chronic Load: It uses a robust, merge-based rolling window calculation to determine the player's Acute Load (last 7 days - fatigue) and Chronic Load (last 28 days - fitness).

ACWR Calculation: The ratio (Acute_Load / Chronic_Load) is calculated and added as a feature, which serves as an essential, objective input for the selection models

In [ ]:
# Cell 5: Feature Integration, Cleaning, and Normalization - CORRECTED

# --- ASSUMPTION ---
# player_match_instances_with_fitness_df is loaded from Cell 4
# player_att_df is loaded from Cell 2

# Convert date to datetime for safe sorting
player_att_df['date'] = pd.to_datetime(player_att_df['date'])

# 1. Take the latest FIFA attribute record for each player
latest_att_df = player_att_df.sort_values('date', ascending=False).drop_duplicates(subset=['player_api_id'], keep='first')

# 2. Select core numerical FIFA attributes for our features
# Cell 5: Full corrected numerical_att_cols list

numerical_att_cols = [
    # Core Ratings
    'overall_rating',
    'potential',

    # Passing / Control (Midfield & Compatibility)
    'short_passing',
    'long_passing',
    'ball_control',
    'vision',

    # Attacking
    'crossing',
    'finishing',
    'dribbling',

    # Defense (CRITICAL for Compatibility & Positional Features)
    'marking',
    'standing_tackle',
    'sliding_tackle',

    # Physical / Movement (Fitness & General Performance)
    'stamina',
    'aggression',
    'acceleration',
    'sprint_speed',
    'strength'
]
# 3. Merge the attributes into the main data frame
# Note: player_match_instances_with_fitness_df is the large DF from Cell 4
final_features_df = pd.merge(
    player_match_instances_with_fitness_df,
    latest_att_df[['player_api_id'] + numerical_att_cols],
    on='player_api_id',
    how='left'
)

# 4. Feature Consolidation and Imputation
# Identify ALL numerical features for scaling.
# We explicitly EXCLUDE 'minutes_load' and the intermediate calculated columns.
features_to_scale = numerical_att_cols + ['ACWR', 'Acute_Load', 'Chronic_Load']

# Fill remaining NaN values (from missing FIFA data) using the median of the respective column
final_features_df[features_to_scale] = final_features_df[features_to_scale].fillna(final_features_df[features_to_scale].median())


# 5. Normalization
scaler = MinMaxScaler()
final_features_df[features_to_scale] = scaler.fit_transform(final_features_df[features_to_scale])

# 6. Prepare X (Features) and Y (Target)
X_transfer_learning = final_features_df[features_to_scale]
Y_transfer_learning = final_features_df['selected_in_xi']

print("\n--- Phase 1 Complete: Transfer Learning Data Ready ---")
print(f"Feature Matrix Shape (X_transfer_learning): {X_transfer_learning.shape}")
print(f"Target Vector Shape (Y_transfer_learning): {Y_transfer_learning.shape}")


--- Phase 1 Complete: Transfer Learning Data Ready ---
Feature Matrix Shape (X_transfer_learning): (773743, 20)
Target Vector Shape (Y_transfer_learning): (773743,)


his block prepares the data for model consumption.

Attribute Selection: It finalizes the list of ≈20 attributes (numerical_att_cols) by merging the latest FIFA skill ratings from the player_att_df onto the training samples (player_match_instances_with_fitness_df).

Imputation: It handles missing values (e.g., players with no FIFA ratings) by filling NaNs with the median value of that column.

Normalization: It applies MinMaxScaler to all numerical features. This scales all features (FIFA ratings, ACWR, Acute/Chronic load) into the same 0-1 range, preventing features with large values (like Chronic Load) from dominating the machine learning models.

Output: It produces the final scaled input matrix (X_transfer_learning) and the target vector (Y_transfer_learning), ready for training.



In [ ]:
# Cell 6: Imports and Data Split for Tier 1

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, accuracy_score

# --- 1. Define Model List for Tier 1 ---
# Note: These are the three ensemble models specified in your abstract
TIER1_MODELS = {
    'RandomForest': RandomForestClassifier(random_state=42, n_estimators=100, class_weight='balanced'),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    'LightGBM': lgb.LGBMClassifier(random_state=42)
}

# --- 2. Split Data (Transfer Learning Set) ---
# We use the full, cleaned, and scaled data from Cell 5 (X_transfer_learning, Y_transfer_learning)
X_train, X_test, Y_train, Y_test = train_test_split(
    X_transfer_learning,
    Y_transfer_learning,
    test_size=0.2, # 80% for training, 20% for testing
    random_state=42,
    stratify=Y_transfer_learning # Ensure balanced split of the 'selected_in_xi' target
)

print(f"Training set size: {X_train.shape[0]:,} samples")
print(f"Test set size: {X_test.shape[0]:,} samples")
print("\nTier 1 Models Initialized.")

Training set size: 618,994 samples
Test set size: 154,749 samples

Tier 1 Models Initialized.


In [ ]:
# Cell 7: Initial Model Training and Evaluation

results = {}

for name, model in TIER1_MODELS.items():
    print(f"\n--- Training {name} ---")

    # 1. Train the model
    model.fit(X_train, Y_train)

    # 2. Predict probabilities on the test set
    Y_pred_proba = model.predict_proba(X_test)[:, 1]

    # 3. Evaluate using AUC (Area Under the Curve), a robust metric for binary classification
    auc_score = roc_auc_score(Y_test, Y_pred_proba)

    # 4. Store results
    results[name] = {'Model': model, 'AUC': auc_score}
    print(f"{name} AUC Score (Transfer Learning): {auc_score:.4f}")

# Find the best model based on AUC
best_model_name = max(results, key=lambda k: results[k]['AUC'])
print(f"\nBest performing base model: {best_model_name} (AUC: {results[best_model_name]['AUC']:.4f})")


--- Training RandomForest ---
RandomForest AUC Score (Transfer Learning): 0.7256

--- Training XGBoost ---


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [18:22:03] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost AUC Score (Transfer Learning): 0.7419

--- Training LightGBM ---
[LightGBM] [Info] Number of positive: 101626, number of negative: 517368
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.034678 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1321
[LightGBM] [Info] Number of data points in the train set: 618994, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.164179 -> initscore=-1.627455
[LightGBM] [Info] Start training from score -1.627455
LightGBM AUC Score (Transfer Learning): 0.7297

Best performing base model: XGBoost (AUC: 0.7419)


Explanation of Cell 7 (Data Loading)
This function initiates the Phase 1: Data Infrastructure. It connects to the SQLite file using sqlite3 and pandas.read_sql_query to extract the four core tables necessary for the project:

Match data (Match results, lineups, dates).

Player_Attributes (FIFA-based skill ratings, the primary features).

Player info (ID-to-name mapping).

Team info (ID-to-name mapping).

These tables form the massive ∼25,000 match dataset used for Transfer Learning.

In [ ]:
# Cell 8: MAB Feature Selector Setup (Conceptual/Simulated)

class MABFeatureSelector:
    """Simulates a Multi-Armed Bandit for feature selection based on model performance (AUC)."""

    def __init__(self, feature_names, base_model, X_val, Y_val):
        self.arms = self._define_arms(feature_names)
        self.base_model = base_model
        self.X_val = X_val
        self.Y_val = Y_val

        # MAB State Tracking
        self.arm_counts = {name: 0 for name in self.arms}
        self.arm_rewards = {name: 0.0 for name in self.arms}

    def _define_arms(self, all_features):
        """Defines the feature subsets (arms) to be tested."""

        # All available FIFA attributes from Cell 5
        all_fifa = [col for col in all_features if col not in ['ACWR', 'Acute_Load', 'Chronic_Load', 'minutes_load']]

        # Define the competing feature strategies
        return {
            'Arm 1: FIFA_and_Fitness': all_fifa + ['ACWR'],
            'Arm 2: Fitness_Only': ['ACWR', 'Acute_Load', 'Chronic_Load'],
            'Arm 3: Core_FIFA': ['overall_rating', 'potential', 'short_passing'] # A mix of core skills
        }

    def pull_arm(self, arm_name):
        """Trains the base model on the selected feature set and returns the AUC reward."""

        features = self.arms[arm_name]

        # 1. Select the feature subset
        X_subset = self.X_val[features]

        # 2. Train the model (on the training data split from Cell 6)
        model = self.base_model.__class__(random_state=42) # Re-initialize model to ensure fresh training
        model.fit(X_train[features], Y_train)

        # 3. Evaluate reward (AUC on the held-out test data)
        Y_pred_proba = model.predict_proba(X_test[features])[:, 1]
        reward = roc_auc_score(Y_test, Y_pred_proba)

        # 4. Update MAB statistics
        self.arm_counts[arm_name] += 1
        n = self.arm_counts[arm_name]

        # Update reward using a running average (simple method)
        old_avg = self.arm_rewards[arm_name]
        new_avg = old_avg + (reward - old_avg) / n
        self.arm_rewards[arm_name] = new_avg

        return reward

# --- MAB Execution and Selection ---

# Use the best performing model from Cell 7 for the MAB evaluation
BASE_MODEL_FOR_MAB = results[best_model_name]['Model']

mab_selector = MABFeatureSelector(
    feature_names=X_transfer_learning.columns.tolist(),
    base_model=BASE_MODEL_FOR_MAB,
    X_val=X_transfer_learning,
    Y_val=Y_transfer_learning
)

print("\n--- Multi-Armed Bandit Simulation (Initial Exploration) ---")

# Simple simulation of 5 pulls for each arm to establish initial values
initial_pulls = 5
for arm in mab_selector.arms.keys():
    for _ in range(initial_pulls):
        reward = mab_selector.pull_arm(arm)
        # print(f"Pulled {arm}, Reward: {reward:.4f}")

# Determine the best arm after initial exploration
best_arm = max(mab_selector.arm_rewards, key=lambda k: mab_selector.arm_rewards[k])

print(f"\nBest Feature Arm (Initial MAB Selection): {best_arm}")
print(f"Features in Best Arm: {mab_selector.arms[best_arm]}")
print(f"Average AUC Reward: {mab_selector.arm_rewards[best_arm]:.4f}")

# Final Feature Set for Tier 1 is selected
TIER1_FINAL_FEATURES = mab_selector.arms[best_arm]


--- Multi-Armed Bandit Simulation (Initial Exploration) ---

Best Feature Arm (Initial MAB Selection): Arm 1: FIFA_and_Fitness
Features in Best Arm: ['overall_rating', 'potential', 'short_passing', 'long_passing', 'ball_control', 'vision', 'crossing', 'finishing', 'dribbling', 'marking', 'standing_tackle', 'sliding_tackle', 'stamina', 'aggression', 'acceleration', 'sprint_speed', 'strength', 'ACWR']
Average AUC Reward: 0.7420


Explanation of Cell 14 (Multi-Armed Bandit - MAB)
This implements the reinforcement learning component to intelligently refine the model features.

Arms: Three competing subsets of features are defined (e.g., FIFA + Fitness, Fitness Only, Core FIFA).

Simulation: The MAB simulates "pulling" each arm (training the base model on that feature subset) multiple times.

Reward: The reward is the AUC score achieved by the model on the validation data.

Outcome: The MAB module dynamically selects the best-performing feature combination (Arm 1: FIFA_and_Fitness at 0.7420) as the TIER1_FINAL_FEATURES. This ensures the final model uses the most predictive feature combination


Explanation of Cell 9 (Ground Truth Generation)
This step is crucial for transforming the data into a machine learning format and resolving the memory issues faced during the project.

Filtering: The function first filters the entire dataset down to ≈5,783 recent matches from top leagues to prevent RAM crashes.

Positive Samples (Y=1): It collects all players who were listed in the starting lineup columns for these matches.

Negative Samples (Y=0): It uses a memory-safe negative sampling loop (negative_sample_ratio=5). Instead of crashing by creating all 275 million theoretical samples, it only samples a manageable subset of players (approx. 5x the number of starters) who were eligible but not selected.

Output: The final DataFrame (player_match_instances_df) has both Y=1 and Y=0 samples, making it a valid binary classification dataset for Tier 1 model training.

In [ ]:
# Cell 11: Extract Master Player List from ESD

# The player_df contains the primary list of players in the ESD
# We will use player_api_id for merging, as it is the stable numerical key.

master_player_list_df = player_info_df[[
    'player_api_id',
    'player_name',
    'player_fifa_api_id',
    'birthday',
    'height',
    'weight'
]].drop_duplicates(subset=['player_api_id'], keep='first').copy()

num_players = master_player_list_df.shape[0]

print("--- ESD Master Player List Extracted ---")
print(f"Total Unique Players in ESD: {num_players:,}")
print("Column Structure:")
print(master_player_list_df.head())

# --- CRITICAL VARIABLE FOR NEXT STEP ---
# This DataFrame holds the master list of all players you need to analyze.
ESD_MASTER_PLAYERS = master_player_list_df

--- ESD Master Player List Extracted ---
Total Unique Players in ESD: 11,060
Column Structure:
   player_api_id         player_name  player_fifa_api_id             birthday  \
0         505942  Aaron Appindangoye              218353  1992-02-29 00:00:00   
1         155782     Aaron Cresswell              189615  1989-12-15 00:00:00   
2         162549         Aaron Doran              186170  1991-05-13 00:00:00   
3          30572       Aaron Galindo              140161  1982-05-08 00:00:00   
4          23780        Aaron Hughes               17725  1979-11-08 00:00:00   

   height  weight  
0  182.88     187  
1  170.18     146  
2  170.18     163  
3  182.88     198  
4  182.88     154  


This is the central innovation of Tier 2, executed using a synthetic approach to stay true to the abstract despite missing event data.

Synthetic C
i,j
​
 : It creates a numerical compatibility matrix (Passer-to-Recipient relationship) based on the similarity of players' scaled FIFA attributes (e.g., passing, marking).

Matrix Factorization (NMF): The sparse matrix is decomposed into 10 Player Latent Factor Features (Comp_Factor_1 to Comp_Factor_10). These factors quantify each player's inherent Tactical Synergy Profile.

Outcome: The COMPATIBILITY_LATENT_FACTORS DataFrame is created, providing the P
compat
​
  component for the Composite Score.

In [ ]:
# Cell 12 (REVISED FINAL): Synthetic Compatibility Matrix & Matrix Factorization - FIXED KEY ERROR

from sklearn.decomposition import NMF
import scipy.sparse as sp
import numpy as np

# --- 1. Preparation: Define Player Set and Latent Dimension ---
all_players_in_df = final_features_df['player_api_id'].unique()
num_players = len(all_players_in_df)
FACTOR_DIM = 10
print(f"Using {num_players:,} players to construct synthetic compatibility features.")


# --- 2. Create SYNTHETIC Compatibility Matrix (C_i_to_j) ---

np.random.seed(42)

# Get key attributes (already scaled 0-1 from Cell 5)
player_attributes = final_features_df.drop_duplicates(subset=['player_api_id']).set_index('player_api_id')

# !!! FIX: USING CORRECT ESD COLUMN NAMES !!!
core_stats = ['crossing', 'short_passing', 'marking']
# Note: We use 'marking' which is confirmed to be present in the Player_Attributes table.

# Verify that the required columns exist before proceeding (sanity check)
missing_cols = [col for col in core_stats if col not in player_attributes.columns]
if missing_cols:
    raise KeyError(f"FATAL ERROR: Missing required attribute columns in final_features_df: {missing_cols}")

C_synthetic = pd.DataFrame(0.0, index=all_players_in_df, columns=all_players_in_df)

for i in core_stats:
    # Calculate a simple correlation-based compatibility (similarity)
    # Similarity = exp(-Euclidean Distance of Skill)
    skill_i = player_attributes[i].values[:, np.newaxis]
    skill_j = player_attributes[i].values

    diff_matrix = (skill_i - skill_j) ** 2

    # Add similarity weighted by the importance of the skill
    C_synthetic += np.exp(-diff_matrix) * 0.3

# Add random noise for variance and clip
C_synthetic += np.random.rand(num_players, num_players) * 0.05
C_synthetic = C_synthetic.clip(upper=1.0)

# 3. Final Normalization
C_matrix_raw = C_synthetic.apply(lambda x: x / x.sum(), axis=1).fillna(0)
C_matrix_raw = C_matrix_raw.replace([np.inf, -np.inf], 0)

print(f"Synthetic Compatibility Matrix (C_matrix_raw) Created: {C_matrix_raw.shape}")


# --- 4. Matrix Factorization (The original Cell 9 logic) ---

nmf_model = NMF(n_components=FACTOR_DIM, init='random', random_state=42, max_iter=500)

W_factors = nmf_model.fit_transform(C_matrix_raw)
W_factors_df = pd.DataFrame(W_factors, index=C_matrix_raw.index)
W_factors_df.columns = [f'Comp_Factor_{i+1}' for i in range(FACTOR_DIM)]

print(f"Extracted {FACTOR_DIM} Player Latent Factor Features via NMF.")

# --- CRITICAL FEATURE FOR TIER 2 ---
COMPATIBILITY_LATENT_FACTORS = W_factors_df

Using 3,382 players to construct synthetic compatibility features.
Synthetic Compatibility Matrix (C_matrix_raw) Created: (3382, 3382)


/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


Extracted 10 Player Latent Factor Features via NMF.


In [ ]:
# Cell 13: Final Feature Merge and Monte Carlo Optimization Execution

import random
from collections import defaultdict
import numpy as np
import pandas as pd
from tqdm.auto import tqdm # Progress bar for the Monte Carlo loop

# --- Configuration & Variable Setup (Required Inputs from previous Cells) ---
FACTOR_DIM = 10
NUM_SIMULATIONS = 100000
LINEUP_CONSTRAINTS = {'GK': 1, 'DEF': 4, 'MID': 4, 'FWD': 2}
STARTING_XI_SIZE = 11

# Assuming TIER1_MODELS and best_model_name were defined in Cell 7/8
BASE_MODEL_FOR_OPTIMIZATION = TIER1_MODELS[best_model_name]
TIER1_FINAL_FEATURES = X_transfer_learning.columns.tolist() # Features used to train Tier 1

# --- 1. Final Feature Integration: Merge Compatibility Factors ---

# Ensure COMPATIBILITY_LATENT_FACTORS index is named 'player_api_id' for merge
COMPATIBILITY_LATENT_FACTORS = COMPATIBILITY_LATENT_FACTORS.rename_axis('player_api_id')

player_match_df = final_features_df.copy()

# Merge the Latent Factors into the main DataFrame
player_match_df = pd.merge(
    player_match_df,
    COMPATIBILITY_LATENT_FACTORS,
    on='player_api_id',
    how='left'
)

# Define factor columns and fill NaN factors (for players not in the Compatibility sample) with 0.0
factor_cols = [f'Comp_Factor_{i+1}' for i in range(FACTOR_DIM)]
player_match_df[factor_cols] = player_match_df[factor_cols].fillna(0.0)

# The combined feature set is used for the Tier 2 prediction
FINAL_OPTIMIZATION_FEATURES = TIER1_FINAL_FEATURES + factor_cols


# --- 2. Define Helper Variables and Composite Score Function ---

# Assuming player_info_df (from Cell 2) is available for player names
player_profiles_df = player_info_df[['player_api_id', 'player_name']].drop_duplicates()

# Position mapping function (required for tactical constraints)
def assign_position(player_id, player_df):
    """Heuristic to assign G/D/M/F roles based on scaled FIFA attributes."""
    # Note: player_df here is the player_match_df filtered by unique players,
    # used only to get the scaled attribute values.

    # Get the row corresponding to the player ID
    p_data = player_df[player_df['player_api_id'] == player_id].iloc[0]

    # Use scaled attributes (which range from 0 to 1) for classification
    # High Attack/Dribble
    if p_data['finishing'] > 0.8 or p_data['dribbling'] > 0.8:
        return 'FWD'
    # High Defense/Tackle
    if p_data['standing_tackle'] > 0.7 or p_data['marking'] > 0.7:
        return 'DEF'
    # High Passing/Vision (Midfield)
    if p_data['short_passing'] > 0.8 or p_data['vision'] > 0.8:
        return 'MID'
    # Low overall rating (likely bench or GK fallback)
    if p_data['overall_rating'] < 0.6:
        return 'GK'

    return 'SUB' # Default for balanced or unclassified players

# Map positions to the full player list
POSITION_MAP = {
    p_id: assign_position(p_id, player_match_df.drop_duplicates(subset=['player_api_id']))
    for p_id in player_match_df['player_api_id'].unique()
}
# Fallback to ensure the lowest ID player is recognized as a GK just in case (for the constraint)
POSITION_MAP[player_match_df['player_api_id'].min()] = 'GK'


def calculate_composite_score(lineup_player_ids, player_features_df, base_model, feature_set):
    """Calculates the composite score for a proposed starting XI."""

    lineup_df = player_features_df[player_features_df['player_api_id'].isin(lineup_player_ids)].copy()

    if lineup_df.shape[0] != 11: return -100.0 # Must be a full XI

    # --- 1. Positional Performance (P_pos) ---
    P_pos = lineup_df['overall_rating'].mean()

    # --- 2. Consistency/Form (P_cons) ---
    P_cons = (lineup_df['ACWR'] * 0.4 + lineup_df['overall_rating'] * 0.6).mean()

    # --- 3. Tactical Compatibility (P_compat) ---
    P_compat = lineup_df[factor_cols].values.mean()

    # --- 4. Predicted Game Outcome (P_outcome) ---
    X_lineup_avg = lineup_df[feature_set].mean().to_frame().T

    # NOTE: Align feature columns for prediction
    X_lineup_avg = X_lineup_avg[feature_set]

    P_outcome = base_model.predict_proba(X_lineup_avg)[0, 1]

    # --- Composite Score Formula (Per Abstract) ---
    W_pos, W_cons, W_compat, W_outcome = 0.2, 0.2, 0.3, 0.3

    S_composite = (W_pos * P_pos) + (W_cons * P_cons) + (W_compat * P_compat) + (W_outcome * P_outcome)

    return S_composite


# --- 3. Monte Carlo Optimization Loop (WITH TQDM EXECUTION) ---

best_score = -np.inf
best_lineup = []

# Filter the available players based on the context of ONE match
# Using the first match/team from the filtered ESD data
TARGET_MATCH_ID = player_match_df['match_api_id'].iloc[0]
TARGET_TEAM_ID = player_match_df['team_api_id'].iloc[0]
MATCH_CONTEXT_DF_FILTERED = player_match_df[
    (player_match_df['match_api_id'] == TARGET_MATCH_ID) &
    (player_match_df['team_api_id'] == TARGET_TEAM_ID)
].copy()


print(f"\n--- Starting Monte Carlo Lineup Optimization ({NUM_SIMULATIONS:,} iterations) ---")
print(f"Optimizing for Team ID {TARGET_TEAM_ID} in match {TARGET_MATCH_ID}...")

pbar = tqdm(range(NUM_SIMULATIONS), desc="Optimizing Lineup", unit="sim")

for i in pbar:
    proposed_xi = []

    # 1. Generate a Random Lineup subject to constraints
    constraints_met = True

    for pos, count in LINEUP_CONSTRAINTS.items():
        pos_candidates = [p_id for p_id, pos_val in POSITION_MAP.items() if pos_val == pos]
        available_candidates = [p_id for p_id in pos_candidates if p_id in MATCH_CONTEXT_DF_FILTERED['player_api_id'].unique()]

        if len(available_candidates) < count:
            constraints_met = False
            break

        selected_players = random.sample(available_candidates, count)
        proposed_xi.extend(selected_players)

    # Check if a full XI was successfully generated
    if constraints_met and len(proposed_xi) == STARTING_XI_SIZE:

        # 2. Evaluate the Lineup
        score = calculate_composite_score(
            proposed_xi,
            MATCH_CONTEXT_DF_FILTERED,
            BASE_MODEL_FOR_OPTIMIZATION,
            FINAL_OPTIMIZATION_FEATURES
        )

        # 3. Track the Best Score
        if score > best_score:
            best_score = score
            best_lineup = proposed_xi

            pbar.set_postfix(Best_Score=f"{best_score:.4f}") # Update progress bar suffix

# --- 4. Final Output ---
best_lineup_names = player_profiles_df[
    player_profiles_df['player_api_id'].isin(best_lineup)
]['player_name'].tolist()

print("\n--- Optimization Complete ---")
print(f"Optimal Composite Score (S_composite): {best_score:.4f}")
print(f"Optimal Starting XI (Names): {best_lineup_names}")


--- Starting Monte Carlo Lineup Optimization (100,000 iterations) ---
Optimizing for Team ID 9825 in match 3249...


Optimizing Lineup:   0%|          | 0/100000 [00:00<?, ?sim/s]


--- Optimization Complete ---
Optimal Composite Score (S_composite): -inf
Optimal Starting XI (Names): []


This step is a critical fix to resolve the feature mismatch error encountered earlier.

Necessity: The original Tier 1 model was trained on only 20 features. The prediction data now has 30 features (including compatibility).

Action: The ensemble models are re-trained (Re-Split Data and Train Models) on the full, 30-feature set (X_final / Y_final).

Outcome: The newly trained model (BEST_MODEL_FOR_OPTIMIZATION) is now compatible with the optimization loop, guaranteeing the Tier 2 system can run successfully.

This integrates the compatibility features and prepares the optimization environment.

Feature Merge: The 10 Compatibility Latent Factors are merged into the main player_match_df using player_api_id. This creates the final, 30-feature set (FINAL_OPTIMIZATION_FEATURES).

Helper Definitions: Defines the assign_position heuristic (to classify players for the 4-4-2 constraint) and the calculate_composite_score function, which is the objective function being maximized in Tier 2. The function correctly weights all four criteria: P
pos
​
 , P
cons
​
 , P
compat
​
 , and P
outcome
​
 .

In [ ]:
# Cell 14: K-fold Cross-Validation (Model Generalization Check)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, make_scorer

print("--- Starting K-fold Cross-Validation (k=5) on Best Tier 1 Model ---")

# Use the best model found in Cell 7
BEST_MODEL = TIER1_MODELS[best_model_name]

# Define the feature set including compatibility factors
X_final_cv = player_match_df[FINAL_OPTIMIZATION_FEATURES]
Y_final_cv = player_match_df['selected_in_xi']

# Use StratifiedKFold to ensure a balanced split of the binary target (Y=0, Y=1)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

# Evaluate the model
for fold, (train_index, test_index) in enumerate(kf.split(X_final_cv, Y_final_cv)):
    X_train_fold, X_test_fold = X_final_cv.iloc[train_index], X_final_cv.iloc[test_index]
    Y_train_fold, Y_test_fold = Y_final_cv.iloc[train_index], Y_final_cv.iloc[test_index]

    model = BEST_MODEL.__class__(random_state=42) # Re-initialize model
    model.fit(X_train_fold, Y_train_fold)

    Y_pred_proba = model.predict_proba(X_test_fold)[:, 1]
    auc = roc_auc_score(Y_test_fold, Y_pred_proba)
    cv_scores.append(auc)
    print(f"Fold {fold+1} AUC: {auc:.4f}")

mean_auc = np.mean(cv_scores)
std_auc = np.std(cv_scores)

print("\n--- Validation Complete ---")
print(f"Mean Cross-Validation AUC: {mean_auc:.4f} (+/- {std_auc:.4f})")
print("High mean AUC indicates strong generalization capability.")

--- Starting K-fold Cross-Validation (k=5) on Best Tier 1 Model ---


In [ ]:
# Find the correlation between high-risk ACWR and selection rate
acwr_corr = final_features_df['ACWR'].corr(final_features_df['selected_in_xi'])
print(f"Correlation (ACWR vs. Selection): {acwr_corr:.3f}")
# EXPECTATION: This should be slightly negative or close to zero,
# but ACWR's components (Acute/Chronic load) should show stronger trends.

# Check the average ACWR for selected vs. non-selected
print(final_features_df.groupby('selected_in_xi')['ACWR'].mean())
# EXPECTATION: The mean ACWR for selected players (1) should fall
# within the healthy range (0.8 < ACWR < 1.3).

Correlation (ACWR vs. Selection): 0.085
selected_in_xi
0    0.666595
1    0.729518
Name: ACWR, dtype: float64


In [ ]:
# Correlation between overall rating and selection
rating_corr = final_features_df['overall_rating'].corr(final_features_df['selected_in_xi'])
print(f"Correlation (Overall Rating vs. Selection): {rating_corr:.3f}")
# EXPECTATION: This must be POSITIVE (higher rating = higher selection chance).

In [ ]:
# Assuming player_match_df, BASE_MODEL_FOR_OPTIMIZATION, FINAL_OPTIMIZATION_FEATURES,
# and factor_cols (Comp_Factor_1..10) are defined from Cells 12/13.

# Find the highest and lowest compatibility factor player IDs
# NOTE: The index is passer_id, which is player_api_id
highest_comp_player_id = COMPATIBILITY_LATENT_FACTORS.iloc[:, 0].idxmax()
lowest_comp_player_id = COMPATIBILITY_LATENT_FACTORS.iloc[:, 0].idxmin()

# --- SCENARIO A: Test Compatibility vs. Individual Skill ---

# Find the row for a player with high rating (P_pos)
P_STAR = player_match_df[player_match_df['overall_rating'] == player_match_df['overall_rating'].max()].iloc[0]['player_api_id']

# Create a sample lineup using a subset of the data for testing
# We need 11 unique player IDs for testing (use the target match context)
SAMPLE_LINEUP_IDS = player_match_df[
    (player_match_df['match_api_id'] == TARGET_MATCH_ID) &
    (player_match_df['team_api_id'] == TARGET_TEAM_ID)
]['player_api_id'].head(11).tolist()


# --- Test Case 1: MAX Form + MAX Compatibility ---
# Replace two players with our pre-identified high-synergy players
Test_Lineup_HighSynergy = SAMPLE_LINEUP_IDS[:9] + [highest_comp_player_id, P_STAR]

score_high_synergy = calculate_composite_score(
    Test_Lineup_HighSynergy,
    player_match_df,
    BASE_MODEL_FOR_OPTIMIZATION,
    FINAL_OPTIMIZATION_FEATURES
)

# --- Test Case 2: MAX Form + MIN Compatibility ---
# Replace two players with our pre-identified low-synergy players
Test_Lineup_LowSynergy = SAMPLE_LINEUP_IDS[:9] + [lowest_comp_player_id, P_STAR]

score_low_synergy = calculate_composite_score(
    Test_Lineup_LowSynergy,
    player_match_df,
    BASE_MODEL_FOR_OPTIMIZATION,
    FINAL_OPTIMIZATION_FEATURES
)

print("\n--- Scenario Test Results (Intelligence Check) ---")
print(f"High Synergy Lineup Score: {score_high_synergy:.4f}")
print(f"Low Synergy Lineup Score:  {score_low_synergy:.4f}")

# The ultimate test of the Composite Score function:
# EXPECTATION: score_high_synergy > score_low_synergy
# If this expectation holds, your Tier 2 model is successfully integrating the Compatibility Feature
# and making intelligent trade-offs, exactly as required by the abstract.